# Merging different LAMMPS trajectory files
A single LAMMPS run is sometimes split in multiple dump trajectory files, for example, if many runs are needed due to short walltime.
This can be an issue since the `trajectory` class can only read one at a time.
Moreover, it may not be smart to try to read 50GB of trajectory files at once.
In that case, each dump file can be read and saved independently, and merged afterwards using `xarray`

This current script assumes that you provide the files in the order of the run, ie. dump0 is at lower timesteps than dump1.

The same concept can be applied to bond dumps

In [2]:
from vast_pea.lammps_traj import trajectory

# Path and dump files to merge
path = "../../data/BER/merge/"
files = ['afd_ber_325K.dump0','afd_ber_325K.dump1','afd_ber_325K.dump2']

## Notice how they have overlapping timesteps.
For overlapping timesteps, we will only keep those of later runs

In [ ]:
for filename in files:
    traj = trajectory(path=path,
                      pattern=filename,
                      filter_type="include",
                      filter=[13], # we will only read sulfur atoms for simplicity
                      verbose=False
    )
    print("Timesteps of file {}: {}".format(filename,traj.timesteps))

Timesteps of file afd_ber_325K.dump0: [1000000 1000010 1000020]
Timesteps of file afd_ber_325K.dump1: [1000030 1000040 1000050 1000060]
Timesteps of file afd_ber_325K.dump2: [1000050 1000060 1000070 1000080 1000090 1000100]


## Now let's merge those trajectories into a single one

In [ ]:
atoms = []

# here we read the files directly since they are small, but for larger ones, it would be advisable to save them first, then merge them
for filename in files:
    traj = trajectory(path=path,
                      pattern=filename,
                      filter_type="include",
                      filter=[13], # we will only read sulfur atoms for simplicity
                      verbose=False
    )
    atoms.append(traj.get_atoms())

# the merged atoms Dataset is saved in new_atoms
# we initialize it by merging the last trajectory read with the second last
new_atoms = atoms[-1].combine_first(
    atoms[-2].where(
        atoms[-2].ts<atoms[-1].ts[0], drop=True # and here we keep only timesteps that are not already present in new_atoms
        )
    )

# then we continue this for all of the other trajectories (in this case only one)
for i in range(len(atoms)-2):
    new_atoms = new_atoms.combine_first(atoms[-3-i].where(atoms[-3-i].ts<atoms[-2-i].ts[0], drop=True))

traj.set_atoms(new_atoms) # overwrite the atoms in the last trajectory read. This will also update some properties of the trajectory class such as the number of timesteps

# we could then save the new traj object to a netCDF file. In the case where we were restore saved netCDF files of filtered trajectories, using the override argument enables adding the filters to the file, since they weren't restored by the restore method

## And let's check the resulting trajectory

In [7]:
traj.timesteps

array([1000000, 1000010, 1000020, 1000030, 1000040, 1000050, 1000060,
       1000070, 1000080, 1000090, 1000100])

In [8]:
traj.get_atoms()

<xarray.Dataset> Size: 536kB
Dimensions:  (ts: 11, id: 752, comp: 3)
Coordinates:
  * ts       (ts) int64 88B 1000000 1000010 1000020 ... 1000080 1000090 1000100
  * id       (id) int64 6kB 12 15 40 43 68 71 ... 45333 45358 45361 45386 45389
  * comp     (comp) <U1 12B 'x' 'y' 'z'
Data variables:
    mol      (ts, id) float64 66kB 1.0 1.0 1.0 1.0 1.0 ... 1.0 1.0 1.0 1.0 1.0
    type     (ts, id) float64 66kB 13.0 13.0 13.0 13.0 ... 13.0 13.0 13.0 13.0
    x        (ts, id) float64 66kB 7.533 5.659 59.08 -19.23 ... 22.35 12.21 13.5
    y        (ts, id) float64 66kB 27.85 27.75 50.49 50.95 ... 59.9 18.2 17.43
    z        (ts, id) float64 66kB 35.37 34.45 11.5 10.48 ... -14.79 1.065 2.404
    ix       (ts, id) float64 66kB 4.0 4.0 5.0 6.0 5.0 ... 5.0 6.0 6.0 6.0 6.0
    iy       (ts, id) float64 66kB -2.0 -2.0 -3.0 -3.0 ... -2.0 -3.0 -3.0 -3.0
    iz       (ts, id) float64 66kB -7.0 -7.0 -7.0 -7.0 ... -6.0 -6.0 -7.0 -7.0
    bounds   (ts, comp) float64 264B 80.0 80.0 80.0 80.0 ... 80.0 80.0 80.0 80.0